In [1]:
# =============================================
# IMPORT
# =============================================
from utility.logger         import aiLogger
from keras.callbacks        import EarlyStopping, ReduceLROnPlateau
from model.componentModel   import mainModel
from model.tepSystemModel   import SystemModel
from preProcessingDataTEP   import PreProcessData
# from prettytable            import PrettyTable

from keras.optimizers       import Adam
from utility.matpilotDraw   import MatPlot
from sklearn.preprocessing  import MinMaxScaler

import tensorflow           as tf
import pandas               as pd
import matplotlib.pyplot    as plt
import numpy                as np
import os, warnings
import random
import json


2025-12-01 18:25:58.869162: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-12-01 18:26:00.667821: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [2]:
# ---------------------------
# Loading config file
# ---------------------------
with open('utility/modelConfig.json', 'r') as configFile:
    configData = json.load(configFile)
config = configData["tep"]  

# ---------------------------
# Set debug level and 
# ---------------------------
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '1'  # or any {'0', '1', '2'}
warnings.filterwarnings('ignore') 

os.environ["CUDA_VISIBLE_DEVICES"] = "0" 
np.random.seed(1)
random.seed(1)
tf.random.set_seed(1)

outputDir               = os.path.expanduser(config["dataDir"])
totalTraject            = 100
normalizeHI             = True
showRedundantShutdown   = False

In [3]:
# ==============================
# Config and data preprocessing
# ==============================
preData             = PreProcessData(config=config)
rawData             = preData.loadDataFromH5Format(dir              = outputDir, 
                                                   name             = "data/teps.h5",
                                                   case             = "all", 
                                                   reduceSampleRate = True, 
                                                   removeHigherThr  = False)

withRULData         = preData.addRULAndEnvironment(rawData)

norData             = preData.normalizeDataEachModes(rawData    =  withRULData,
                                                     headline   =   preData.headline["allWithoutLabelNoBias"] + ["rul"])    # Normalize the data

Process:122300||2025-12-01 18:26:07,470-INFO [preProcessingDataTEP.py:loadDataFromH5Format():358]:Loading data from H5 files ...
Process:122300||2025-12-01 18:26:10,582-INFO [preProcessingDataTEP.py:loadDataFromH5Format():385]:Finish loading data from H5 files at case all...
Process:122300||2025-12-01 18:26:10,912-INFO [preProcessingDataTEP.py:addRULAndEnvironment():434]:Adding rul successfully...
Process:122300||2025-12-01 18:26:10,913-INFO [preProcessingDataTEP.py:normalizeDataEachModes():801]:Normalizing the data ...


In [ ]:
# -*- coding: utf-8 -*-
"""
Per-Client TEST Evaluation for SystemModel (revised for dict-based splits)
--------------------------------------------------------------------------------

What's inside:
- MatPlot: comprehensive plotting helper from utility.plot
- _to_numpy, _safe_feature_mean: helpers
- test_client_split_with_model: runs inference on a single split dict and computes metrics
- visualize_client_predictions_matplot: MatPlot-like plotting for predictions
- evaluate_clients_2_3_4_on_test: evaluate all three clients on test splits and summarize
- test_client_trajectory_based: trajectory-based testing with HI normalization
- visualize_trajectories_subplot: subplot visualization for all trajectories

CRITICAL UPDATES FOR DICT-BASED SPLITS:
- Adapted to work with dict format from build_env_clients_2_3_4
- Each split dict has keys: {"X": [X_allSensor, X_time, X_env], "y": y_lastRUL}
- X_time and X_env are FULL SEQUENCES (shape: [N, seq_len, dim])
- We extract the LAST timestep for model input: X_time[:, -1, :] and X_env[:, -1, :]
- y_lastRUL is already the last timestep RUL with shape [N, 1] - no need to reshape
"""

import os
import numpy as np
from typing import Any, Dict, List, Optional, Tuple
# from tqdm import tqdm

# ==============================================================================
# MatPlot helper (import from utility.plot)
# ==============================================================================
from utility.matpilotDraw   import MatPlot

# ==============================================================================
# Utilities
# ==============================================================================
def _to_numpy(x: Any) -> np.ndarray:
    """
    Convert TF/PyTorch/NumPy inputs to a NumPy array safely.
    """
    if hasattr(x, "numpy"):
        return x.numpy()
    try:
        import torch  # noqa: F401
        import torch as _torch
        if isinstance(x, _torch.Tensor):  # type: ignore[attr-defined]
            return x.detach().cpu().numpy()
    except Exception:
        pass
    return np.asarray(x)


def _safe_feature_mean(arr: np.ndarray, n_samples: int) -> np.ndarray:
    """
    Compute mean over feature/channel dimension robustly.
    Many HI arrays are [N, D, ...]. We want a per-sample scalar trend.
    """
    try:
        return arr[:n_samples].mean(axis=1)
    except Exception:
        arr2 = arr[:n_samples].reshape(n_samples, -1)
        return arr2.mean(axis=1)


# ==============================================================================
# Core: single-split evaluation (adapted for dict-based splits from build_env_clients_2_3_4)
# ==============================================================================
def test_client_split_with_model(
    split_data: Dict[str, Any],
    client_id: Any,
    vaeSys_initiator: Any,
    config: Dict[str, Any],
    split_name: str = "test",
) -> Dict[str, Any]:
    """
    Run SystemModel on a single client's split (dict format from build_env_clients_2_3_4).

    CRITICAL: This function is adapted to work with the NEW dict-based split format:
    - split_data["X"] = [X_allSensor, X_time, X_env]
      * X_allSensor: (N, seq_len, n_features) - full sensor sequences (NORMALIZED)
      * X_time: (N, seq_len, time_dim) - full time sequences (NORMALIZED)
      * X_env: (N, seq_len, env_dim) - full environment sequences (NORMALIZED)
    - split_data["y"] = y_lastRUL
      * y_lastRUL: (N, 1) - TRUE RUL at the LAST timestep (UNNORMALIZED from original data)

    The model expects inputs with last timestep for time and env:
    - model_input = [X_allSensor, X_time[:, -1, :], X_env[:, -1, :]]

    Args:
        split_data: dict with keys "X" (list of [X_allSensor, X_time, X_env]) and "y" (y_lastRUL)
        client_id: identifier for the client (e.g., 2, 3, 4)
        vaeSys_initiator: model instance with method getHIandRUL(input=[seqX, time, env])
        config: dict with model configuration
        split_name: name of the split for logging (e.g., "test", "val", "train")

    Returns:
        results: dict with numpy arrays:
            - 'hi_reac', 'hi_sep', 'hi_strip', 'hi_sys', 'hi_matrix'
            - 'rul_pred', 'rul_true', 'time_values'
            - 'metrics' (mae/rmse/smape/r2)
    """
    print("=" * 80)
    print(f"Testing Client {client_id} - {split_name.upper()} Split")
    print("=" * 80)

    # Extract data from split_data dict
    X_list = split_data.get("X", [])
    y_true = split_data.get("y", np.array([]))

    if len(X_list) != 3:
        raise ValueError(f"Expected X to contain 3 arrays [X_allSensor, X_time, X_env], got {len(X_list)}")

    # Extract arrays from dict format
    # CRITICAL: X_time and X_env are FULL SEQUENCES from build_env_clients_2_3_4
    X_allSensor = _to_numpy(X_list[0])  # shape: (N, seq_len, n_features) - normalized sensor features
    X_time_full = _to_numpy(X_list[1])  # shape: (N, seq_len, time_dim) - normalized time sequences
    X_env_full = _to_numpy(X_list[2])   # shape: (N, seq_len, env_dim) - normalized env sequences
    y_true = _to_numpy(y_true)          # shape: (N, 1) - TRUE RUL at last timestep (unnormalized)

    n_samples = X_allSensor.shape[0]

    print(f"Total samples: {n_samples}")
    print(f"Model configuration:")
    print(f"  - Sequence length: {config.get('seqLength')}")
    print(f"  - Input features:  {config.get('inReac')}")
    print(f"  - HI selection dim:{config.get('hiSelectionDim', 120)}")

    if n_samples == 0:
        print("Warning: No samples in this split. Returning empty results.")
        return {
            "hi_reac": np.array([]),
            "hi_sep": np.array([]),
            "hi_strip": np.array([]),
            "hi_sys": np.array([]),
            "hi_matrix": np.array([]),
            "rul_pred": np.array([]),
            "rul_true": np.array([]),
            "time_values": np.array([]),
            "metrics": {"mae": np.nan, "rmse": np.nan, "smape": np.nan, "r2": np.nan}
        }

    # Extract LAST timestep for time and env to match model expectations
    # This is CRITICAL: the model expects (N, time_dim) and (N, env_dim) for last timestep
    X_time_last = X_time_full[:, -1, :]  # shape: (N, time_dim) - last timestep
    X_env_last = X_env_full[:, -1, :]    # shape: (N, env_dim) - last timestep

    print(f"Input shapes:")
    print(f"  - X_allSensor (full sequences): {X_allSensor.shape}")
    print(f"  - X_time (full sequences): {X_time_full.shape}")
    print(f"  - X_env (full sequences): {X_env_full.shape}")
    print(f"  - X_time_last (last timestep for model): {X_time_last.shape}")
    print(f"  - X_env_last (last timestep for model): {X_env_last.shape}")
    print(f"  - y_true (last timestep RUL): {y_true.shape}")

    # Run model inference with LAST timestep for time and env
    model_input = [X_allSensor, X_time_last, X_env_last]
    hiReac, hiSep, hiStrip, hiSystem, predictedRul, hiMatrixOutput = vaeSys_initiator.getHIandRUL(input=model_input)

    # Convert to numpy
    hi_reac = _to_numpy(hiReac)
    hi_sep = _to_numpy(hiSep)
    hi_strip = _to_numpy(hiStrip)
    hi_sys = _to_numpy(hiSystem)
    hi_matrix = _to_numpy(hiMatrixOutput)
    y_pred = _to_numpy(predictedRul)

    # Use LAST timestep time values for visualization (already extracted)
    time_vals = X_time_last.reshape(-1) if X_time_last.size > 0 else np.array([])

    # Flatten y arrays for metric computation
    y_true_flat = y_true.reshape(-1)
    y_pred_flat = y_pred.reshape(-1)

    # Sanity checks
    for name, arr in [("time", time_vals), ("rul_true", y_true_flat), ("rul_pred", y_pred_flat)]:
        if np.isnan(arr).any() or np.isinf(arr).any():
            raise ValueError(f"[NaN/Inf] Found in {name}")

    print(f"\nProcessed samples: {y_true_flat.shape[0]}")
    print(f"Time range: {time_vals.min():.3f} -> {time_vals.max():.3f}")
    print(f"True RUL  : min={y_true_flat.min():.3f}  max={y_true_flat.max():.3f}")
    print(f"Pred RUL  : min={y_pred_flat.min():.3f}  max={y_pred_flat.max():.3f}")

    # Compute metrics (using sMAPE instead of MAPE to avoid division by zero)
    mae = float(np.mean(np.abs(y_pred_flat - y_true_flat)))
    rmse = float(np.sqrt(np.mean((y_pred_flat - y_true_flat) ** 2)))

    # sMAPE: Symmetric Mean Absolute Percentage Error (avoids division by zero)
    # sMAPE = 100 * mean(|y_true - y_pred| / (|y_true| + |y_pred|))
    denominator = np.abs(y_true_flat) + np.abs(y_pred_flat) + 1e-8
    smape = float(np.mean(np.abs(y_true_flat - y_pred_flat) / denominator) * 200.0)

    ss_res = float(np.sum((y_true_flat - y_pred_flat) ** 2))
    ss_tot = float(np.sum((y_true_flat - np.mean(y_true_flat)) ** 2) + 1e-12)
    r2 = float(1.0 - ss_res / ss_tot)

    print("\n" + "=" * 80)
    print(f"Evaluation Metrics ({split_name.upper()} - Client {client_id})")
    print("=" * 80)
    print(f"  MAE:   {mae:.4f}")
    print(f"  RMSE:  {rmse:.4f}")
    print(f"  sMAPE: {smape:.2f}%")
    print(f"  R2:    {r2:.4f}")

    results = {
        "hi_reac": hi_reac,
        "hi_sep": hi_sep,
        "hi_strip": hi_strip,
        "hi_sys": hi_sys,
        "hi_matrix": hi_matrix,
        "rul_pred": y_pred_flat,
        "rul_true": y_true_flat,
        "time_values": time_vals,
        "metrics": {"mae": mae, "rmse": rmse, "smape": smape, "r2": r2}
    }
    return results


# ==============================================================================
# Trajectory-based testing with HI normalization
# ==============================================================================
def test_client_trajectory_based(
    norData: Any,
    preData: Any,
    vaeSys_initiator: Any,
    environment_idx: int = 2,
    normalize_hi: bool = True,
    show_redundant_shutdown: bool = False,
) -> Dict[str, Any]:
    """
    Test client using trajectory-based approach with HI normalization.

    Args:
        norData: normalized data array
        preData: preprocessor instance with methods getTensorData and normalizeHIInGeneral
        vaeSys_initiator: model instance with method getHIandRUL
        environment_idx: environment index to test (e.g., 2 for client 2)
        normalize_hi: whether to normalize HI values
        show_redundant_shutdown: whether to show redundant shutdown information

    Returns:
        results: dict containing HI values, RUL predictions, and time arrays for all trajectories
    """
    print("=" * 80)
    print(f"Trajectory-Based Testing for Environment {environment_idx}")
    print("=" * 80)

    # Extract data for the specified environment
    data = norData[environment_idx]
    trajectory_id = data["Id"]
    reactor = data[preData.headline["reactor"]]
    separator = data[preData.headline["separator"]]
    stripper = data[preData.headline["stripper"]]
    all_sensor = data[preData.headline["allWithoutLabelNoBias"]]
    rul = data["rul"]
    env = data["environmentID"]
    time_traj = data["Time"]

    # Get total number of trajectories
    total_traject = len(np.unique(trajectory_id))
    print(f"Total trajectories in environment {environment_idx}: {total_traject}")

    # Initialize storage dictionaries
    hi_env = {}
    rul_env = {}
    hi_env["Reac"] = []
    rul_env["Reac"] = []
    hi_env["Sep"] = []
    rul_env["Sep"] = []
    hi_env["Strip"] = []
    rul_env["Strip"] = []
    hi_env["Sys"] = []
    rul_env["Sys"] = []
    hi_env["multipliedMatrix"] = []
    time_draw = []

    # Process each trajectory with progress bar (suppressing individual prints)
    print(f"Processing {total_traject} trajectories...")
    for idx in tqdm(range(total_traject), desc=f"Client {environment_idx}", ncols=100):
        get_at_idx = idx + 1

        # Get tensor data for current trajectory
        seq_x_all_sensor_train, seq_x_rul_train, seq_x_train_env, seq_time = preData.getTensorData(
            dataList=[all_sensor, rul, env, time_traj],
            trajectId=trajectory_id,
            shuffle=False,
            getAtIdx=get_at_idx,
            seqLength=preData.config["seqLength"],
            overlap=preData.config["overLap"]
        )

        seq_time_train = seq_time[:, -1, :]

        # Prepare model input
        x_predict = [seq_x_all_sensor_train, seq_time[:, -1, :], seq_x_train_env[:, -1, :]]

        # Get HI and RUL predictions
        hi_reac, hi_sep, hi_strip, hi_system, predicted_rul, hi_matrix_output = vaeSys_initiator.getHIandRUL(input=x_predict)

        # Store results
        time_draw.append(seq_time_train)
        hi_env["Reac"].append(hi_reac)
        hi_env["Sep"].append(hi_sep)
        hi_env["Strip"].append(hi_strip)
        hi_env["Sys"].append(hi_system)
        hi_env["multipliedMatrix"].append(hi_matrix_output)
        rul_env["Sys"].append(predicted_rul)

    # Normalize HI if requested
    if normalize_hi:
        [hi_env["Reac"], hi_env["Sep"], hi_env["Strip"], hi_env["Sys"]], time_draw, rul_env["Sys"], hi_env["multipliedMatrix"] = preData.normalizeHIInGeneral(
            inData=[hi_env["Reac"], hi_env["Sep"], hi_env["Strip"], hi_env["Sys"]],
            time=time_draw,
            rul=rul_env["Sys"],
            multipliedMt=hi_env["multipliedMatrix"],
            indexToShutdown=2,
            showRedundantShutdown=show_redundant_shutdown
        )

    print(f"✓ Completed trajectory-based testing for environment {environment_idx}")
    print(f"  Total trajectories processed: {len(time_draw)}")

    results = {
        "hi_env": hi_env,
        "rul_env": rul_env,
        "time_draw": time_draw,
        "environment_idx": environment_idx,
        "total_trajectories": total_traject
    }

    return results


# ==============================================================================
# Trajectory subplot visualization
# ==============================================================================
def visualize_trajectories_subplot(
    trajectory_results: Dict[str, Any],
    save_dir: str = "checkpoint/client_predictions/",
    num_row: int = 20,
    num_col: int = 5,
) -> None:
    """
    Visualize trajectory-based results using subplot for all trajectories.

    Args:
        trajectory_results: results dict from test_client_trajectory_based
        save_dir: directory to save plots
        num_row: number of rows in subplot grid
        num_col: number of columns in subplot grid
    """
    os.makedirs(save_dir, exist_ok=True)

    hi_env = trajectory_results["hi_env"]
    rul_env = trajectory_results["rul_env"]
    time_draw = trajectory_results["time_draw"]
    environment_idx = trajectory_results["environment_idx"]

    print("\n" + "=" * 80)
    print(f"Creating Subplot Visualizations for Environment {environment_idx}")
    print("=" * 80)

    # Plot HI components subplot
    plot1 = MatPlot()
    plot1.subPlotMultiImg(
        inY=[hi_env["Reac"], hi_env["Sep"], hi_env["Strip"], hi_env["Sys"]],
        inX=time_draw,
        color=['green', 'blue', 'yellow', 'red'],
        numRow=num_row,
        numCol=num_col
    )
    print(f"✓ Created HI components subplot (Reactor, Separator, Stripper, System)")

    # Plot RUL subplot
    plot2 = MatPlot()
    plot2.subPlotMultiImg(
        inY=[rul_env["Sys"]],
        inX=time_draw,
        color=['g'],
        numRow=num_row,
        numCol=num_col
    )
    print(f"✓ Created RUL predictions subplot")

    print("=" * 80)


# ==============================================================================
# MatPlot-like visualization
# ==============================================================================
def visualize_client_predictions_matplot(
    results: Dict[str, Any],
    client_id: Any,
    split: str = "test",
    save_dir: str = "checkpoint/client_predictions/",
    also_plot_error: bool = True,
):
    """
    Draw prediction plots using MatPlot:
      - One plot of True vs Pred RUL over time
      - One plot of error vs sample index
      - Component HI plots
      - System HI plots
    """
    mpl = MatPlot()
    os.makedirs(save_dir, exist_ok=True)

    time_vals = results.get("time_values", np.array([]))
    y_true = results.get("rul_true", np.array([]))
    y_pred = results.get("rul_pred", np.array([]))
    hi_reac = results.get("hi_reac", np.array([]))
    hi_sep = results.get("hi_sep", np.array([]))
    hi_strip = results.get("hi_strip", np.array([]))
    hi_sys = results.get("hi_sys", np.array([]))

    if time_vals.size == 0 or y_true.size == 0 or y_pred.size == 0:
        print(f"No data to visualize for client {client_id} {split} split.")
        return

    # Sort by time for better visualization
    sort_idx = np.argsort(time_vals)
    time_sorted = time_vals[sort_idx]
    y_true_sorted = y_true[sort_idx]
    y_pred_sorted = y_pred[sort_idx]

    # Plot True vs Pred RUL using draw2DMultipleY
    mpl.draw2DMultipleY(
        xCoor=time_sorted,
        yCoor=[y_true_sorted, y_pred_sorted],
        yLabel="RUL",
        xLabel="Time",
        title=f"Client {client_id} — {split.upper()} — True vs Pred RUL",
        saveFig=True,
        direction=save_dir,
        savedName=f"client_{client_id}_{split}_true_vs_pred",
        useLegend=True,
        legend=["True RUL", "Pred RUL"],
        color=["blue", "green"]
    )
    print(f"✓ Saved: {os.path.join(save_dir, f'client_{client_id}_{split}_true_vs_pred.png')}")

    if also_plot_error:
        # Plot error vs sample index
        err = y_pred - y_true
        x_idx = np.arange(err.shape[0], dtype=np.float32)
        mpl.draw2D(
            xCoor=x_idx,
            yCoor=err,
            yLabel="Error (Pred - True)",
            xLabel="Sample Index",
            title=f"Client {client_id} — {split.upper()} — RUL Prediction Error",
            saveFig=True,
            direction=save_dir,
            savedName=f"client_{client_id}_{split}_error",
            color="red"
        )
        print(f"✓ Saved: {os.path.join(save_dir, f'client_{client_id}_{split}_error.png')}")

    # Plot Component HIs
    if hi_reac.size > 0 and hi_sep.size > 0 and hi_strip.size > 0:
        hi_reac_mean = _safe_feature_mean(hi_reac, hi_reac.shape[0])
        hi_sep_mean = _safe_feature_mean(hi_sep, hi_sep.shape[0])
        hi_strip_mean = _safe_feature_mean(hi_strip, hi_strip.shape[0])

        mpl.draw2DMultipleY(
            xCoor=time_sorted,
            yCoor=[hi_reac_mean[sort_idx], hi_sep_mean[sort_idx], hi_strip_mean[sort_idx]],
            yLabel="Component HIs",
            xLabel="Cycle",
            title=f"Client {client_id} — {split.upper()} — Component HIs",
            saveFig=True,
            direction=save_dir,
            savedName=f"client_{client_id}_{split}_component_his",
            useLegend=True,
            legend=["Reactor", "Separator", "Stripper"],
            color=["blue", "green", "orange"],
            yLim=[0, 1.1]
        )
        print(f"✓ Saved: {os.path.join(save_dir, f'client_{client_id}_{split}_component_his.png')}")

    # Plot System HI
    if hi_sys.size > 0:
        hi_sys_mean = _safe_feature_mean(hi_sys, hi_sys.shape[0])

        mpl.draw2D(
            xCoor=time_sorted,
            yCoor=hi_sys_mean[sort_idx],
            yLabel="System HI",
            xLabel="Cycle",
            title=f"Client {client_id} — {split.upper()} — System HI",
            saveFig=True,
            direction=save_dir,
            savedName=f"client_{client_id}_{split}_system_hi",
            color="red"
        )
        print(f"✓ Saved: {os.path.join(save_dir, f'client_{client_id}_{split}_system_hi.png')}")

    # Plot All HIs together
    if hi_reac.size > 0 and hi_sep.size > 0 and hi_strip.size > 0 and hi_sys.size > 0:
        mpl.draw2DMultipleY(
            xCoor=time_sorted,
            yCoor=[hi_reac_mean[sort_idx], hi_sep_mean[sort_idx], hi_strip_mean[sort_idx], hi_sys_mean[sort_idx]],
            yLabel="All HIs",
            xLabel="Cycle",
            title=f"Client {client_id} — {split.upper()} — All HIs",
            saveFig=True,
            direction=save_dir,
            savedName=f"client_{client_id}_{split}_all_his",
            useLegend=True,
            legend=["Reactor", "Separator", "Stripper", "System"],
            color=["blue", "green", "orange", "red"],
            yLim=[0, 1.1]
        )
        print(f"✓ Saved: {os.path.join(save_dir, f'client_{client_id}_{split}_all_his.png')}")


# ==============================================================================
# All-clients TEST evaluation and summary table
# ==============================================================================
def evaluate_clients_2_3_4_on_test(
    test_client_2: Dict[str, Any],
    test_client_3: Dict[str, Any],
    test_client_4: Dict[str, Any],
    vaeSys_initiator: Any,
    config: Dict[str, Any],
    save_csv_path: Optional[str] = None,
    visualize_each: bool = True,
    save_dir: str = "checkpoint/client_predictions/",
    # New parameters for trajectory-based testing
    norData: Optional[Any] = None,
    preData: Optional[Any] = None,
    trajectory_based_test: bool = False,
    normalize_hi: bool = True,
    show_redundant_shutdown: bool = False,
    subplot_num_row: int = 20,
    subplot_num_col: int = 5,
) -> Dict[str, Any]:
    """
    Evaluate TEST splits for clients 2, 3, and 4 and summarize.

    CRITICAL: This function is adapted to work with dict-based splits from build_env_clients_2_3_4:
    - Each test_client_* is a dict with keys {"X": [X_allSensor, X_time, X_env], "y": y_lastRUL}
    - X arrays contain FULL SEQUENCES (X_time and X_env have shape [N, seq_len, dim])
    - We extract the last timestep internally for model input

    Args:
        test_client_2: test split dict for client 2
        test_client_3: test split dict for client 3
        test_client_4: test split dict for client 4
        vaeSys_initiator: model instance
        config: model configuration dict
        save_csv_path: optional path to save per-client metrics CSV
        visualize_each: whether to save visualization plots for each client
        save_dir: directory to save plots
        norData: normalized data for trajectory-based testing
        preData: preprocessor instance for trajectory-based testing
        trajectory_based_test: whether to perform trajectory-based testing
        normalize_hi: whether to normalize HI in trajectory-based testing
        show_redundant_shutdown: whether to show redundant shutdown info
        subplot_num_row: number of rows for subplot visualization
        subplot_num_col: number of columns for subplot visualization

    Returns:
        summary: dict with per-client DataFrame, macro/micro averages, and detailed results
    """
    print("\n" + "#" * 80)
    print("# Evaluating Clients 2, 3, 4 on TEST Split")
    print("# Using dict-based splits from build_env_clients_2_3_4")
    print("#" * 80)

    test_splits = {
        2: test_client_2,
        3: test_client_3,
        4: test_client_4,
    }

    rows: List[Dict[str, Any]] = []
    detailed_results: Dict[Any, Dict[str, Any]] = {}
    trajectory_results: Dict[Any, Dict[str, Any]] = {}

    for client_id, split_data in test_splits.items():
        try:
            res = test_client_split_with_model(
                split_data=split_data,
                client_id=client_id,
                vaeSys_initiator=vaeSys_initiator,
                config=config,
                split_name="test",
            )
            detailed_results[client_id] = res

            m = res.get("metrics", {})
            mae = float(m.get("mae", np.nan))
            rmse = float(m.get("rmse", np.nan))
            smape = float(m.get("smape", np.nan))
            r2 = float(m.get("r2", np.nan))
            n = int(res["rul_true"].shape[0]) if isinstance(res.get("rul_true"), np.ndarray) else 0

            rows.append({
                "client_id": client_id,
                "n_samples": n,
                "MAE": mae,
                "RMSE": rmse,
                "sMAPE(%)": smape,
                "R2": r2
            })

            if visualize_each:
                visualize_client_predictions_matplot(
                    results=res,
                    client_id=client_id,
                    split="test",
                    save_dir=save_dir,
                    also_plot_error=True,
                )

            # Perform trajectory-based testing if requested
            if trajectory_based_test and norData is not None and preData is not None:
                print(f"\n{'=' * 80}")
                print(f"Performing Trajectory-Based Testing for Client {client_id}")
                print(f"{'=' * 80}")

                traj_res = test_client_trajectory_based(
                    norData=norData,
                    preData=preData,
                    vaeSys_initiator=vaeSys_initiator,
                    environment_idx=client_id,
                    normalize_hi=normalize_hi,
                    show_redundant_shutdown=show_redundant_shutdown,
                )
                trajectory_results[client_id] = traj_res

                # Visualize trajectory-based results
                visualize_trajectories_subplot(
                    trajectory_results=traj_res,
                    save_dir=save_dir,
                    num_row=subplot_num_row,
                    num_col=subplot_num_col,
                )

        except Exception as e:
            print(f"[WARN] Client {client_id} evaluation failed: {e}")
            rows.append({
                "client_id": client_id,
                "n_samples": 0,
                "MAE": np.nan,
                "RMSE": np.nan,
                "sMAPE(%)": np.nan,
                "R2": np.nan
            })

    # Summary table
    import pandas as pd
    df = pd.DataFrame(rows).sort_values(by="client_id").reset_index(drop=True)

    macro_mae = df["MAE"].mean(skipna=True)
    macro_rmse = df["RMSE"].mean(skipna=True)
    macro_smape = df["sMAPE(%)"].mean(skipna=True)
    macro_r2 = df["R2"].mean(skipna=True)

    # Micro averages by pooling samples
    pooled_pred, pooled_true = [], []
    for client_id, res in detailed_results.items():
        y_hat = res.get("rul_pred")
        y_true = res.get("rul_true")
        if isinstance(y_hat, np.ndarray) and isinstance(y_true, np.ndarray) and y_hat.size > 0 and y_true.size > 0:
            pooled_pred.append(y_hat.reshape(-1))
            pooled_true.append(y_true.reshape(-1))

    if pooled_pred:
        pooled_pred = np.concatenate(pooled_pred, axis=0)
        pooled_true = np.concatenate(pooled_true, axis=0)
        micro_mae = float(np.mean(np.abs(pooled_pred - pooled_true)))
        micro_rmse = float(np.sqrt(np.mean((pooled_pred - pooled_true) ** 2)))
        denominator = np.abs(pooled_true) + np.abs(pooled_pred) + 1e-8
        micro_smape = float(np.mean(np.abs(pooled_true - pooled_pred) / denominator) * 200.0)
        ss_res = float(np.sum((pooled_true - pooled_pred) ** 2))
        ss_tot = float(np.sum((pooled_true - np.mean(pooled_true)) ** 2) + 1e-12)
        micro_r2 = float(1.0 - ss_res / ss_tot)
    else:
        micro_mae = micro_rmse = micro_smape = micro_r2 = np.nan

    # Pretty print
    print("\n" + "=" * 80)
    print("Per-Client TEST Metrics")
    print("=" * 80)
    with np.printoptions(precision=4, suppress=True):
        print(df.to_string(index=False))

    print("\n" + "-" * 80)
    print("Macro Average (unweighted across clients):")
    print(f"  MAE={macro_mae:.4f}  RMSE={macro_rmse:.4f}  sMAPE%={macro_smape:.2f}  R2={macro_r2:.4f}")
    print("Micro Average (weighted by samples):")
    print(f"  MAE={micro_mae:.4f}  RMSE={micro_rmse:.4f}  sMAPE%={micro_smape:.2f}  R2={micro_r2:.4f}")
    print("-" * 80)

    if save_csv_path:
        os.makedirs(os.path.dirname(save_csv_path), exist_ok=True)
        df.to_csv(save_csv_path, index=False)
        print(f"✓ Saved per-client TEST metrics to: {save_csv_path}")

    summary = {
        "per_client_df": df,
        "macro_avg": {"mae": macro_mae, "rmse": macro_rmse, "smape": macro_smape, "r2": macro_r2},
        "micro_avg": {"mae": micro_mae, "rmse": micro_rmse, "smape": micro_smape, "r2": micro_r2},
        "detailed_results": detailed_results,
        "trajectory_results": trajectory_results,
    }
    return summary


In [ ]:
config["component"] = "Sys"
componentModelPath  = "checkpoint/bestModel200Epoch/"
vaeSys              = SystemModel(config = config, componentModelPath = componentModelPath)
vaeSys.build(input_shape=[(None, config["seqLength"], config["inReac"]), (None, 1, config["inReac"]), (None, 1)])
vaeSys.summary()


# modelDirComp1 = outputDir + componentModelPath + "bestModelReac"
# modelDirComp2 = outputDir + componentModelPath + "bestModelSep"
# modelDirComp3 = outputDir + componentModelPath + "bestModelStrip"
# modelDirSys   = outputDir + componentModelPath + "bestModelSys"
# vaeSys.load_weights(modelDirComp1, modelDirComp2, modelDirComp3, modelDirSys)

In [7]:
# -*- coding: utf-8 -*-
"""
Federated LoRA on subclassed Keras SystemModel — Keras 3 safe (no add_weight after build).
What this script does (unchanged functionality, industrial refactor):
- Loads the base SystemModel strictly via a mirror pipeline (.h5/joblib on Drive)
- Attaches LoRA EXTERNALLY (no state added to built layers) and optimizes LoRA only
- Runs FedAvg across env-2/3/4 clients (optional FedProx + server momentum)
- Early stopping on global validation (weighted MSE)
- TWO-STAGE procedure:
    1) Quick baseline evaluation (NO LoRA, NO training) on all variants → select best k*
    2) Full LoRA training only on the best variant k*
- Enforces RUL non-negativity via an output clamping (ReLU) wrapper
- Adds series-system "minimum HI" bias: softmin layer override + series-min regularization
- Speed flags: XLA JIT, optional mixed precision, reduced logging
- Saves best LoRA to:
    /content/drive/MyDrive/Duc_an_XAI_edge_cooperation/kun/checkpoint/HI_model_update/lora/variant_{k}/best_lora.npz

Industrial changes vs. the provided draft:
- All tunables are centralized in `FLLoraConfig` (config_FL_Lora)
- A single high-level `run_federated_lora(system_config, config_FL_Lora)` orchestrates the pipeline
- The function returns:
    (best_model_with_lora_mounted, best_lora_numpy_state_dict)
  where the model is constructed for the best variant and has the best LoRA already loaded.

IMPORTANT
- The code assumes your project provides `SystemModel` and that global numpy dicts for data
  (train_client_2/3/4, val_client_2/3/4, test_client_2/3/4, norData, preData) exist,
  exactly as in your original environment. This refactor deliberately preserves data flow.
"""

import os
import re
import sys
import json
import time
import glob
import copy
import shutil
import pathlib
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

import numpy as np
import tensorflow as tf
from tensorflow.keras import backend as K

# =============================================================================
# Paths & project setup (kept as constants to preserve your environment)
# =============================================================================
HI_UPDATE_ROOT   = "./checkpoint/flCkpt/HI_model_update/"
ORIGINALS_ROOT   = "./checkpoint/flCkpt/bestModel_5epoch/"
TMP_MIRROR_ROOT  = "./checkpoint/flCkpt/_tmp_variant_mirror/"
LORA_SAVE_ROOT   = os.path.join(HI_UPDATE_ROOT, "lora")



os.makedirs(TMP_MIRROR_ROOT, exist_ok=True)
os.makedirs(LORA_SAVE_ROOT, exist_ok=True)

SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

# You must have `SystemModel` importable from your project path inserted above.
# from your_project.models import SystemModel
# In your environment the symbol is expected to be available already.
# SystemModel = globals().get("SystemModel", None)  # pragma: no cover

# SystemModel = SystemModel()

# =============================================================================
# Centralized configuration (config_FL_Lora)
# =============================================================================
@dataclass
class FLLoraConfig:
    """
    Container for all knobs used by the pipeline.
    Defaults match your original script exactly.
    """
    # ---------- Stage 2 training ----------
    global_rounds: int = 500
    local_epochs: int = 2
    local_steps_per_epoch: int = 3

    # ---------- Optimization ----------
    batch_size: int = 512
    lr: float = 5e-4
    weight_decay: float = 1e-4
    patience: int = 10
    min_delta: float = 0.0

    # ---------- LoRA ----------
    lora_r: int = 32
    lora_alpha: float = 64.0
    lora_dropout: float = 0.15
    lora_target_regex: str = r".*"  # match all Dense layers

    # ---------- Federated ----------
    use_fedprox: bool = True
    fedprox_mu: float = 1e-3
    use_server_momentum: bool = True
    server_momentum_beta: float = 0.9
    server_step: float = 1.0

    # ---------- Pre/Post / Label ----------
    use_label_zscore: bool = False  # kept for parity; labels not scaled in this script

    # ---------- Speed / Logs ----------
    use_xla_jit: bool = True
    use_mixed_precision: bool = False
    verbose_local: bool = False
    log_every_n_steps: int = 50
    warmup_after_patch: bool = False

    # ---------- RUL non-negativity ----------
    enforce_rul_nonneg: bool = True

    # ---------- Series-system min HI ----------
    use_softmin_bias: bool = True
    softmin_temp: float = 1.0
    series_min_reg_weight: float = 0.01  # loss weight

    # ---------- Variants source ----------
    use_hi_updates: bool = True  # pick files from *_HI_updates

    # ---------- Softmin attach targets (HI selection/mask) ----------
    softmin_target_regex: str = r".*Selection.*|.*Mask.*"


# =============================================================================
# XLA / mixed precision (applied inside the runner to keep globals clean)
# =============================================================================
def _configure_runtime(cfg: FLLoraConfig):
    if cfg.use_xla_jit:
        tf.config.optimizer.set_jit(True)
        print("[XLA] JIT compilation enabled.")
    if cfg.use_mixed_precision:
        from tensorflow.keras import mixed_precision
        policy = mixed_precision.Policy('mixed_float16')
        mixed_precision.set_global_policy(policy)
        print("[Mixed Precision] Enabled with policy:", policy.name)


# =============================================================================
# Mirror helpers
# =============================================================================
def _comp_dir(root: str, sub: str, use_hi: bool) -> str:
    return os.path.join(root, f"{sub}{'_HI_updates' if use_hi else ''}")

def _ensure_empty_dir(path: str):
    if os.path.isdir(path):
        shutil.rmtree(path)
    os.makedirs(path, exist_ok=True)

def _copy_variant(src_dir: str, dst_dir: str, mapping_with_suffix: dict, idx: int):
    os.makedirs(dst_dir, exist_ok=True)
    for expected_name, base_stem in mapping_with_suffix.items():
        src = os.path.join(src_dir, f"{base_stem}_{idx}.h5")
        dst = os.path.join(dst_dir, f"{expected_name}.h5")
        if not os.path.isfile(src):
            raise FileNotFoundError(f"Variant file not found: {src}")
        shutil.copy2(src, dst)

def mirror_and_build_system_model(
                                    variant_idx: int,
                                    use_hi_updates: bool,
                                    system_config: dict
                                ) -> tf.keras.Model:
    """
    Build and load your SystemModel strictly via the mirrored directory pipeline.
    """
    # if SystemModel is None:
    #     raise RuntimeError("SystemModel not found. Ensure your project is on sys.path and imported.")

    K.clear_session()

    # Sources
    SRC_REAC_DIR  = _comp_dir(HI_UPDATE_ROOT, "bestModelReac",  use_hi_updates)
    SRC_SEP_DIR   = _comp_dir(HI_UPDATE_ROOT, "bestModelSep",   use_hi_updates)
    SRC_STRIP_DIR = _comp_dir(HI_UPDATE_ROOT, "bestModelStrip", use_hi_updates)
    SRC_SYS_DIR   = _comp_dir(HI_UPDATE_ROOT, "bestModelSys",   use_hi_updates)

    # Mirror
    DST_REAC_DIR  = os.path.join(TMP_MIRROR_ROOT, "bestModelReac")
    DST_SEP_DIR   = os.path.join(TMP_MIRROR_ROOT, "bestModelSep")
    DST_STRIP_DIR = os.path.join(TMP_MIRROR_ROOT, "bestModelStrip")
    DST_SYS_DIR   = os.path.join(TMP_MIRROR_ROOT, "bestModelSys")

    _ensure_empty_dir(DST_REAC_DIR)
    _ensure_empty_dir(DST_SEP_DIR)
    _ensure_empty_dir(DST_STRIP_DIR)
    _ensure_empty_dir(DST_SYS_DIR)

    COMP_MAPPING = {"encoderWts": "encoderWts", "decoderWts": "decoderWts", "maskWts": "maskWts"}
    SYS_MAPPING  = {"hiAndRulWts": "hiAndRulWts"}

    _copy_variant(SRC_REAC_DIR,  DST_REAC_DIR,  COMP_MAPPING, variant_idx)
    _copy_variant(SRC_SEP_DIR,   DST_SEP_DIR,   COMP_MAPPING, variant_idx)
    _copy_variant(SRC_STRIP_DIR, DST_STRIP_DIR, COMP_MAPPING, variant_idx)
    _copy_variant(SRC_SYS_DIR,   DST_SYS_DIR,   SYS_MAPPING,  variant_idx)

    # Minimal sanity
    required_keys = ["seqLength", "inReac"]
    missing = [k for k in required_keys if k not in system_config]
    if missing:
        raise KeyError(f"Missing required SystemModel config keys: {missing}")

    # Prepare config for SystemModel
    system_config = dict(system_config)  # shallow copy
    system_config["component"] = "Sys"
    system_config.setdefault("hiSelectionDim", 120)
    component_dir_root = TMP_MIRROR_ROOT

    # Instantiate
    vaeSys_contributor = SystemModel(config=system_config, componentModelPath=component_dir_root)

    # Load weights from the mirror
    modelDirComp1 = os.path.join(component_dir_root, "bestModelReac")
    modelDirComp2 = os.path.join(component_dir_root, "bestModelSep")
    modelDirComp3 = os.path.join(component_dir_root, "bestModelStrip")
    modelDirSys   = os.path.join(component_dir_root, "bestModelSys")
    vaeSys_contributor.load_weights(modelDirComp1, modelDirComp2, modelDirComp3, modelDirSys)

    return vaeSys_contributor


# =============================================================================
# Recursive layer traversal (robust)
# =============================================================================
def iter_all_layers(root: tf.keras.Model):
    """Yield all layers recursively, descending into nested tf.keras.Model or `.model` attrs."""
    seen = set()
    def _walk(layer):
        if layer in seen:
            return
        seen.add(layer)
        yield layer
        kids = []
        if hasattr(layer, "layers"):
            kids.extend(layer.layers)
        if hasattr(layer, "model") and isinstance(layer.model, tf.keras.Model):
            kids.append(layer.model)
        for ch in kids:
            for y in _walk(ch):
                yield y
    for l in _walk(root):
        if isinstance(l, tf.keras.layers.Layer):
            yield l


# =============================================================================
# RUL non-negativity: output clamping wrapper
# =============================================================================
class RULClampLayer(tf.keras.layers.Layer):
    """Clamp RUL output to be non-negative via ReLU."""
    def call(self, inputs):
        return tf.nn.relu(inputs)

def wrap_model_with_rul_clamp(model: tf.keras.Model, cfg: FLLoraConfig) -> tf.keras.Model:
    """
    Wrap the model's output with a RUL clamping layer.
    Assumes model returns (possibly tuple/list) where the last element is RUL.
    """
    if not cfg.enforce_rul_nonneg:
        return model

    class RULClampedModel(tf.keras.Model):
        def __init__(self, base_model):
            super().__init__()
            self.base_model = base_model
            self.clamp = RULClampLayer(name="rul_clamp")
        def call(self, inputs, training=None):
            outputs = self.base_model(inputs, training=training)
            if isinstance(outputs, (list, tuple)):
                *rest, rul = outputs
                return (*rest, self.clamp(rul))
            return self.clamp(outputs)

    return RULClampedModel(model)


# =============================================================================
# Series-system minimum HI: softmin bias mechanism
# =============================================================================
def attach_softmin_bias_to_dense(layer: tf.keras.layers.Dense, temp: float):
    """
    For a Dense layer with output_dim=3, override call() to apply softmin to logits.
    This encourages the model to select the minimum HI component.
    """
    if getattr(layer, "_softmin_patched", False):
        return

    units = int(layer.kernel.shape[1])
    if units != 3:
        return  # Only apply to 3-output layers (HI selection mask)

    import types
    original_call = layer.call

    def call_with_softmin(self, inputs, training=None, **kwargs):
        logits = original_call(inputs, training=training, **kwargs)
        # softmin = softmax(-logits / T)
        softmin_weights = tf.nn.softmax(-logits / temp, axis=-1)
        return softmin_weights

    layer.call = types.MethodType(call_with_softmin, layer)
    layer._softmin_patched = True

def attach_softmin_to_model(model: tf.keras.Model, cfg: FLLoraConfig):
    """
    Attach softmin bias mechanism to target Dense layers (e.g., HI selection layers).
    """
    if not cfg.use_softmin_bias:
        return
    pat = re.compile(cfg.softmin_target_regex)
    count = 0
    for lyr in iter_all_layers(model):
        if isinstance(lyr, tf.keras.layers.Dense) and pat.match(lyr.name):
            attach_softmin_bias_to_dense(lyr, temp=cfg.softmin_temp)
            count += 1
    if count > 0:
        print(f"[Softmin] Attached to {count} Dense layer(s).")


# =============================================================================
# EXTERNAL LoRA (Keras 3 safe; no add_weight after build)
# =============================================================================
_LORA_REGISTRY: Dict[tf.keras.layers.Dense, Dict[str, tf.Variable]] = {}

def clear_lora_registry():
    _LORA_REGISTRY.clear()

def attach_lora_to_dense_layer_external(
    layer: tf.keras.layers.Dense, r: int, alpha: float, dropout: float
):
    """
    Create external tf.Variable A/B (not tracked by the layer), override call() to use them.
    Base layer is frozen; only LoRA variables are trained.
    """
    if getattr(layer, "_lora_patched", False):
        return
    if not hasattr(layer, "kernel") or layer.kernel is None:
        raise RuntimeError(f"Dense '{layer.name}' is not built; cannot attach LoRA.")

    in_dim  = int(layer.kernel.shape[0])
    units   = int(layer.kernel.shape[1])
    use_bias = getattr(layer, "use_bias", True)
    act_fn   = layer.activation
    scale    = float(alpha) / float(r) if r > 0 else 0.0
    drop     = tf.keras.layers.Dropout(dropout) if (dropout and dropout > 0) else None

    lora_A = tf.Variable(
        initial_value=tf.keras.initializers.HeNormal()(shape=(in_dim, r), dtype=layer.kernel.dtype),
        trainable=True, dtype=layer.kernel.dtype, name=f"{layer.name}_lora_A")
    lora_B = tf.Variable(
        initial_value=tf.zeros((r, units), dtype=layer.kernel.dtype),
        trainable=True, dtype=layer.kernel.dtype, name=f"{layer.name}_lora_B")

    _LORA_REGISTRY[layer] = {"A": lora_A, "B": lora_B,
                             "scale": tf.constant(scale, dtype=layer.kernel.dtype),
                             "drop": drop, "act": act_fn}

    # Freeze base weights
    layer.trainable = False

    import types
    def call_with_external_lora(self, inputs, training=None, **kwargs):
        x = inputs
        y = tf.linalg.matmul(x, self.kernel)
        if use_bias and getattr(self, "bias", None) is not None:
            y = y + self.bias
        pack = _LORA_REGISTRY[self]
        if r > 0 and float(pack["scale"].numpy()) != 0.0:
            x_in = pack["drop"](x, training=training) if pack["drop"] is not None else x
            lo = tf.linalg.matmul(x_in, pack["A"])
            lo = tf.linalg.matmul(lo, pack["B"])
            y = y + pack["scale"] * lo
        if pack["act"] is not None:
            y = pack["act"](y)
        return y

    layer.call = types.MethodType(call_with_external_lora, layer)
    layer._lora_patched = True

def attach_lora_to_model_external(model: tf.keras.Model, cfg: FLLoraConfig):
    clear_lora_registry()
    pat = re.compile(cfg.lora_target_regex)
    count = 0
    for lyr in iter_all_layers(model):
        if isinstance(lyr, tf.keras.layers.Dense) and pat.match(lyr.name):
            attach_lora_to_dense_layer_external(lyr, r=cfg.lora_r, alpha=cfg.lora_alpha, dropout=cfg.lora_dropout)
            count += 1
    print(f"[LoRA] Attached (external) to {count} Dense layer(s).")

def get_lora_variables_external(model: tf.keras.Model) -> Dict[str, tf.Variable]:
    out = {}
    for lyr, pack in _LORA_REGISTRY.items():
        out[f"{lyr.name}/lora_A"] = pack["A"]
        out[f"{lyr.name}/lora_B"] = pack["B"]
    return out

def lora_state_numpy_external(model: tf.keras.Model) -> Dict[str, np.ndarray]:
    return {k: v.numpy() for k, v in get_lora_variables_external(model).items()}

def load_lora_state_from_numpy_external(model: tf.keras.Model, state: Dict[str, np.ndarray]):
    name2var = get_lora_variables_external(model)
    for k, arr in state.items():
        if k in name2var:
            name2var[k].assign(arr)


# =============================================================================
# Data pipeline (expects your globals: train_client_2/3/4 etc.)
# =============================================================================
def _np_to_tf(x: np.ndarray) -> tf.Tensor:
    return tf.convert_to_tensor(x, dtype=tf.float32)

def _pack_inputs_for_systemmodel(x_all: tf.Tensor, env_id: int) -> Tuple[tf.Tensor, tf.Tensor, tf.Tensor]:
    """Pack inputs as (rawData, seqCycle, env) to match SystemModel."""
    batch = tf.shape(x_all)[0]
    seqCycle = tf.zeros((batch, 1), dtype=tf.float32)
    env = tf.fill([batch, 1], tf.cast(env_id, tf.int32))
    return x_all, seqCycle, env

def make_tf_dataset(client_split: Dict[str, List[np.ndarray]],
                    batch_size: int, shuffle: bool, env_id: int) -> tf.data.Dataset:
    X_all, _, _ = client_split["X"]
    y = client_split["y"].astype(np.float32)
    ds = tf.data.Dataset.from_tensor_slices((_np_to_tf(X_all), _np_to_tf(y)))
    if shuffle:
        ds = ds.shuffle(buffer_size=min(10000, X_all.shape[0]), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.batch(batch_size, drop_remainder=False).prefetch(tf.data.AUTOTUNE)

    def _map(x_all, y_true):
        inputs = _pack_inputs_for_systemmodel(x_all, env_id=env_id)
        return inputs, y_true

    return ds.map(_map, num_parallel_calls=tf.data.AUTOTUNE)

def build_all_client_datasets(batch_size: int):
    g = globals()
    def bundle(eid):
        return {"train": g[f"train_client_{eid}"],
                "val":   g[f"val_client_{eid}"],
                "test":  g[f"test_client_{eid}"]}
    c2, c3, c4 = bundle(2), bundle(3), bundle(4)
    def make_all(c, env_id):
        tr = make_tf_dataset(c["train"], batch_size, True,  env_id)
        va = make_tf_dataset(c["val"],   batch_size, False, env_id)
        te = make_tf_dataset(c["test"],  batch_size, False, env_id)
        n  = c["train"]["y"].shape[0]
        return tr, va, te, n
    return make_all(c2, 2), make_all(c3, 3), make_all(c4, 4)


# =============================================================================
# Loss, metrics, label scaler (kept for parity)
# =============================================================================
def mse_loss(y_pred, y_true):
    return tf.reduce_mean(tf.square(y_pred - y_true))

def series_min_regularization(hi_reac, hi_sep, hi_strip):
    """
    Regularization encouraging series-system minimum HI behavior.
    Penalizes deviation from min(HI_reac, HI_sep, HI_strip).
    """
    hi_stack = tf.stack([hi_reac, hi_sep, hi_strip], axis=-1)  # (B, 3)
    hi_min = tf.reduce_min(hi_stack, axis=-1, keepdims=True)   # (B, 1)
    reg = tf.reduce_mean(tf.square(hi_stack - hi_min))
    return reg

class LabelScaler:
    def __init__(self):
        self.mean = None
        self.std  = None
    def fit(self, y_list: List[np.ndarray]):
        y = np.concatenate(y_list, axis=0).astype(np.float32)
        self.mean = float(np.mean(y))
        self.std  = float(np.std(y) + 1e-8)
    def transform(self, y):
        return (y - self.mean) / self.std
    def inverse(self, y):
        return y * self.std + self.mean

def _extract_rul_from_model_output(outs):
    """SystemModel.call returns many things; RUL is the last."""
    return outs[-1] if isinstance(outs, (list, tuple)) else outs

def _extract_hi_components_from_model_output(outs):
    """
    Attempt to extract HI_reac, HI_sep, HI_strip from model output.
    Expected structure (may vary in your project):
    [recon_reac, recon_sep, recon_strip, HI_reac, HI_sep, HI_strip, ..., RUL]
    """
    if not isinstance(outs, (list, tuple)):
        return None, None, None
    try:
        if len(outs) >= 7:
            hi_reac = outs[3]
            hi_sep  = outs[4]
            hi_strip= outs[5]
            return hi_reac, hi_sep, hi_strip
    except Exception:
        pass
    return None, None, None


# =============================================================================
# Warm-up on a real mini-batch
# =============================================================================
def warmup_with_real_batch(model: tf.keras.Model, ds: tf.data.Dataset, cfg: FLLoraConfig):
    if not cfg.warmup_after_patch:
        return
    print("[WARMUP] Running a small forward pass...")
    t0 = time.time()
    for (inputs, _) in ds.take(1):
        _ = model(inputs, training=False)
        _ = model(inputs, training=True)
    print(f"[WARMUP] Done in {time.time() - t0:.2f}s.")


# =============================================================================
# FedOps
# =============================================================================
def evaluate_global(model: tf.keras.Model, val_datasets: List[tf.data.Dataset],
                    scaler: Optional[LabelScaler]) -> Tuple[float, float, List[float]]:
    total_loss = 0.0
    total_n = 0
    per_client = []
    for ds in val_datasets:
        c_loss = 0.0
        c_n = 0
        for (inputs, y) in ds:
            outs = model(inputs, training=False)
            y_pred = _extract_rul_from_model_output(outs)
            loss = mse_loss(y_pred, y).numpy()
            bs = int(y.shape[0])
            c_loss += loss * bs
            c_n    += bs
        per_client.append(c_loss / max(1, c_n))
        total_loss += c_loss
        total_n    += c_n
    weighted = total_loss / max(1, total_n) if total_n > 0 else float("inf")
    mean_mse = float(np.mean(per_client)) if per_client else float("inf")
    return weighted, mean_mse, per_client

def local_train_one_client(
    model: tf.keras.Model,
    train_ds: tf.data.Dataset,
    client_tag: str,
    round_idx: int,
    global_lora_numpy: Dict[str, np.ndarray],
    cfg: FLLoraConfig,
    scaler: Optional[LabelScaler]
) -> Dict[str, np.ndarray]:
    # broadcast latest global LoRA
    load_lora_state_from_numpy_external(model, global_lora_numpy)

    name2var  = get_lora_variables_external(model)
    lora_vars = list(name2var.values())
    optimizer = tf.keras.optimizers.Adam(learning_rate=cfg.lr)

    for ep in range(1, cfg.local_epochs + 1):
        step = 0

        for (inputs, y) in train_ds.take(cfg.local_steps_per_epoch):
            step += 1
            with tf.GradientTape() as tape:
                outs = model(inputs, training=True)
                y_pred = _extract_rul_from_model_output(outs)

                loss = mse_loss(y_pred, y)

                # Series-min regularization
                if cfg.series_min_reg_weight > 0:
                    hi_reac, hi_sep, hi_strip = _extract_hi_components_from_model_output(outs)
                    if hi_reac is not None and hi_sep is not None and hi_strip is not None:
                        series_reg = series_min_regularization(hi_reac, hi_sep, hi_strip)
                        loss = loss + cfg.series_min_reg_weight * series_reg

                # FedProx on LoRA vars
                if cfg.use_fedprox and cfg.fedprox_mu > 0:
                    prox_term = 0.0
                    for name, var in name2var.items():
                        g = tf.convert_to_tensor(global_lora_numpy[name])
                        prox_term += tf.reduce_sum(tf.square(var - g))
                    loss = loss + 0.5 * cfg.fedprox_mu * prox_term

                # L2 on LoRA vars
                if cfg.weight_decay and cfg.weight_decay > 0:
                    wd_term = tf.add_n([tf.reduce_sum(tf.square(v)) for v in lora_vars]) * cfg.weight_decay
                    loss = loss + wd_term

            grads = tape.gradient(loss, lora_vars)
            grads, _ = tf.clip_by_global_norm(grads, 1.0)
            optimizer.apply_gradients(zip(grads, lora_vars))

            if cfg.verbose_local and (step % cfg.log_every_n_steps == 0):
                print(f"[R{round_idx:03d}|{client_tag}|E{ep}|S{step}] loss={float(loss.numpy()):.6f}")

    return lora_state_numpy_external(model)

def fedavg_numpy(global_state: Dict[str, np.ndarray],
                 client_states: List[Tuple[Dict[str, np.ndarray], int]]) -> Dict[str, np.ndarray]:
    keys = list(global_state.keys())
    if not keys:
        return global_state
    total_n = sum(n for _, n in client_states)
    total_n = max(1, total_n)
    acc = {k: np.zeros_like(global_state[k]) for k in keys}
    for state, n in client_states:
        w = float(n) / total_n
        for k in keys:
            acc[k] += state[k] * w
    return acc

def fedavg_with_momentum(old_global: Dict[str, np.ndarray],
                         new_fedavg: Dict[str, np.ndarray],
                         mom_buf: Dict[str, np.ndarray],
                         beta: float,
                         step: float):
    keys = list(old_global.keys())
    for k in keys:
        delta = new_fedavg[k] - old_global[k]
        mom_buf[k] = beta * mom_buf[k] + (1 - beta) * delta
        old_global[k] = old_global[k] + step * mom_buf[k]
    return old_global, mom_buf


# =============================================================================
# Variant enumeration
# =============================================================================
def find_all_variant_indices(use_hi_updates: bool) -> List[int]:
    folder = _comp_dir(HI_UPDATE_ROOT, "bestModelReac", use_hi_updates)
    paths = glob.glob(os.path.join(folder, "encoderWts_*.h5"))
    idxs = []
    for p in paths:
        m = re.search(r"_(\d+)\.h5$", p)
        if m:
            idxs.append(int(m.group(1)))
    return sorted(set(idxs))


# =============================================================================
# Test evaluation utility (kept as-is; optional for your workflow)
# =============================================================================
def evaluate_variant_on_test(variant_idx: int, use_hi_updates: bool, system_config: dict, cfg: FLLoraConfig):
    """
    Independent test evaluation function, called after training is complete.
    Rebuilds the model, mounts best LoRA, and runs your project's test routine.
    """
    print(f"\n========== Test Evaluation for Variant {variant_idx} ==========")

    save_dir = os.path.join(LORA_SAVE_ROOT, f"variant_{variant_idx}")
    best_lora_path = os.path.join(save_dir, "best_lora.npz")

    if not os.path.exists(best_lora_path):
        print(f"⚠ Best LoRA not found at {best_lora_path}, skipping test evaluation")
        return

    # Rebuild model and load best LoRA
    K.clear_session()
    test_model = mirror_and_build_system_model(variant_idx, use_hi_updates, system_config)
    test_model = wrap_model_with_rul_clamp(test_model, cfg)
    attach_softmin_to_model(test_model, cfg)
    attach_lora_to_model_external(test_model, cfg)

    # Load best LoRA weights
    best_lora_loaded = dict(np.load(best_lora_path))
    load_lora_state_from_numpy_external(test_model, best_lora_loaded)
    print(f"✓ Loaded best LoRA from: {best_lora_path}")

    # Get test data splits from globals
    g = globals()
    test_client_2 = g["test_client_2"]
    test_client_3 = g["test_client_3"]
    test_client_4 = g["test_client_4"]
    norData = g["norData"]
    preData = g["preData"]

    # Your project-specific evaluation entry point (kept intact)
    try:
        summary = evaluate_clients_2_3_4_on_test(  # noqa: F821 (provided in your environment)
            test_client_2=test_client_2,
            test_client_3=test_client_3,
            test_client_4=test_client_4,
            vaeSys_initiator=test_model,
            config=system_config,
            save_csv_path=os.path.join(save_dir, f"test_metrics_variant_{variant_idx}.csv"),
            visualize_each=True,
            save_dir=save_dir,
            norData=norData,
            preData=preData,
            trajectory_based_test=True,
            normalize_hi=True,
            show_redundant_shutdown=False,
            subplot_num_row=20,
            subplot_num_col=5,
        )

        # Save detailed results
        import joblib
        for client_id, res in summary["detailed_results"].items():
            out_pkl = os.path.join(save_dir, f"client_{client_id}_test_results.pkl")
            joblib.dump(res, out_pkl)

        for client_id, traj_res in summary["trajectory_results"].items():
            out_pkl = os.path.join(save_dir, f"client_{client_id}_trajectory_results.pkl")
            joblib.dump(traj_res, out_pkl)

        print(f"✓ Test evaluation completed for variant {variant_idx}")

    except Exception as e:
        print(f"⚠ Test evaluation failed for variant {variant_idx}: {e}")
        import traceback
        traceback.print_exc()


# =============================================================================
# Stage 1: Quick baseline evaluation (NO training, NO LoRA)
# =============================================================================
def quick_baseline_evaluation(
    variant_idx: int,
    use_hi_updates: bool,
    system_config: dict,
    val_loaders: List[tf.data.Dataset],
    cfg: FLLoraConfig
) -> float:
    """
    Evaluate the original model (NO LoRA, NO training) on validation sets.
    Returns weighted validation MSE (lower is better).
    """
    print(f"[Baseline] Evaluating variant {variant_idx}...")

    # Build & load base SystemModel (no LoRA)
    base_model = mirror_and_build_system_model(variant_idx, use_hi_updates, system_config)
    base_model = wrap_model_with_rul_clamp(base_model, cfg)
    attach_softmin_to_model(base_model, cfg)

    # Evaluate on validation sets (no training)
    w_val, m_val, per_client = evaluate_global(base_model, val_loaders, scaler=None)
    print(f"[Baseline] Variant {variant_idx}: Val MSE = {w_val:.6f}")

    # Clean up
    K.clear_session()

    return w_val


# =============================================================================
# Stage 2: Full training on selected best variant
# =============================================================================
def full_train_best_variant(
    variant_idx: int,
    use_hi_updates: bool,
    system_config: dict,
    train_loaders: List[tf.data.Dataset],
    val_loaders: List[tf.data.Dataset],
    train_sizes: List[int],
    client_tags: List[str],
    cfg: FLLoraConfig
) -> Tuple[tf.keras.Model, Dict[str, np.ndarray], float]:
    """
    Full training on the selected best variant with early stopping.
    Returns:
      - model: the base SystemModel with LoRA attached
      - best_lora: numpy dict of the best LoRA state
      - best_val: best weighted validation MSE achieved
    """
    print(f"\n========== Full Training: Variant {variant_idx} ==========")

    # Build & load base SystemModel
    base_model = mirror_and_build_system_model(variant_idx, use_hi_updates, system_config)
    base_model = wrap_model_with_rul_clamp(base_model, cfg)
    attach_softmin_to_model(base_model, cfg)

    # Attach LoRA externally
    attach_lora_to_model_external(base_model, cfg)

    # Optional warm-up
    warmup_with_real_batch(base_model, train_loaders[0], cfg)

    # Label scaler (disabled)
    scaler = None

    # Initialize global LoRA
    global_lora = lora_state_numpy_external(base_model)
    best_lora   = copy.deepcopy(global_lora)
    best_val    = float("inf")
    rounds_no_improve = 0

    server_m = None
    if cfg.use_server_momentum:
        server_m = {k: np.zeros_like(v) for k, v in global_lora.items()}

    save_dir = os.path.join(LORA_SAVE_ROOT, f"variant_{variant_idx}")
    os.makedirs(save_dir, exist_ok=True)

    # Full training rounds
    for rnd in range(1, cfg.global_rounds + 1):
        print(f"\n[Round {rnd:03d}] Local training...")
        client_updates: List[Tuple[Dict[str, np.ndarray], int]] = []

        for ci, train_ds in enumerate(train_loaders):
            lora_after = local_train_one_client(
                base_model, train_ds,
                client_tag=client_tags[ci],
                round_idx=rnd,
                global_lora_numpy=global_lora,
                cfg=cfg,
                scaler=scaler
            )
            client_updates.append((lora_after, train_sizes[ci]))

        # Server aggregation
        fedavg_lora = fedavg_numpy(global_lora, client_states=client_updates)
        if cfg.use_server_momentum:
            global_lora, server_m = fedavg_with_momentum(
                global_lora, fedavg_lora, server_m,
                beta=cfg.server_momentum_beta, step=cfg.server_step
            )
        else:
            global_lora = fedavg_lora

        # Evaluate
        load_lora_state_from_numpy_external(base_model, global_lora)
        w_val, m_val, per_client = evaluate_global(base_model, val_loaders, scaler)
        print(f"[Round {rnd:03d}] Val MSE: {w_val:.6f} (per-client: {[f'{v:.4f}' for v in per_client]})")

        # Early stopping
        if w_val + cfg.min_delta < best_val:
            best_val = w_val
            best_lora = copy.deepcopy(global_lora)
            rounds_no_improve = 0

            # Save best LoRA
            np.savez_compressed(os.path.join(save_dir, "best_lora.npz"), **best_lora)
            meta = {
                "variant_idx": variant_idx,
                "use_hi_updates": use_hi_updates,
                "lora_config": {
                    "r": cfg.lora_r, "alpha": cfg.lora_alpha, "dropout": cfg.lora_dropout,
                    "target_regex": cfg.lora_target_regex
                },
                "train_config": {
                    "global_rounds": cfg.global_rounds, "local_epochs": cfg.local_epochs,
                    "lr": cfg.lr, "weight_decay": cfg.weight_decay,
                    "patience": cfg.patience, "min_delta": cfg.min_delta,
                    "fedprox": cfg.use_fedprox, "fedprox_mu": cfg.fedprox_mu,
                    "server_momentum": cfg.use_server_momentum, "beta": cfg.server_momentum_beta,
                    "local_steps_per_epoch": cfg.local_steps_per_epoch,
                },
                "val_loss_weighted": float(best_val),
                "timestamp": int(time.time()),
            }
            with open(os.path.join(save_dir, "meta.json"), "w") as f:
                json.dump(meta, f, indent=2)
            print(f"[Round {rnd:03d}] ✓ Saved best LoRA (val={best_val:.6f})")
        else:
            rounds_no_improve += 1
            if rounds_no_improve >= cfg.patience:
                print(f"Early stopping at round {rnd} (no improvement for {cfg.patience} rounds).")
                break

    print(f"✓ Finished training variant {variant_idx}. Best val MSE = {best_val:.6f}")

    # Ensure the returned model has the best LoRA state mounted
    load_lora_state_from_numpy_external(base_model, best_lora)
    return base_model, best_lora, best_val


# =============================================================================
# Orchestrator: run the full two-stage pipeline and return the best model + weights
# =============================================================================
def run_federated_lora(system_config: dict, config_FL_Lora: FLLoraConfig) -> Tuple[tf.keras.Model, Dict[str, np.ndarray]]:
    """
    High-level entry point.

    Args
    ----
    system_config : dict
        Configuration for your SystemModel constructor (as used in your project).
        Must include required keys like "seqLength" and "inReac".
    config_FL_Lora : FLLoraConfig
        Centralized hyperparameters & flags.

    Returns
    -------
    best_model : tf.keras.Model
        The SystemModel for the best variant, wrapped (RUL clamp) + softmin bias attached,
        with LoRA externally attached and the best LoRA weights already loaded.
    best_lora_state : Dict[str, np.ndarray]
        Numpy dict of the trained LoRA variables (keys like "<dense_name>/lora_A", ".../lora_B").
    """
    cfg = config_FL_Lora
    _configure_runtime(cfg)

    # Build datasets once from globals
    (c2_train, c2_val, c2_test, n2), (c3_train, c3_val, c3_test, n3), (c4_train, c4_val, c4_test, n4) = \
        build_all_client_datasets(cfg.batch_size)
    train_loaders = [c2_train, c3_train, c4_train]
    val_loaders   = [c2_val,   c3_val,   c4_val]
    train_sizes   = [n2, n3, n4]
    client_tags   = ["env2", "env3", "env4"]

    # Stage 1: Quick baseline evaluation (NO training, NO LoRA)
    print("\n" + "="*80)
    print("STAGE 1: Quick baseline evaluation (NO training, NO LoRA)")
    print("="*80)

    idxs = find_all_variant_indices(use_hi_updates=cfg.use_hi_updates)
    if not idxs:
        raise RuntimeError("No perturbed variants found.")
    print("Found perturbed variant indices:", idxs)

    variant_scores: Dict[int, float] = {}
    for k in idxs:
        val_mse = quick_baseline_evaluation(k, cfg.use_hi_updates, system_config, val_loaders, cfg)
        variant_scores[k] = val_mse

    # Select best variant
    best_variant = min(variant_scores, key=variant_scores.get)
    best_score = variant_scores[best_variant]

    print("\n" + "="*80)
    print("BASELINE EVALUATION RESULTS:")
    for k in sorted(variant_scores.keys()):
        marker = " ← BEST" if k == best_variant else ""
        print(f"  Variant {k}: Val MSE = {variant_scores[k]:.6f}{marker}")
    print("="*80)
    print(f"\n✓ Selected best variant: {best_variant} (Val MSE = {best_score:.6f})")

    # Stage 2: Full LoRA training on best variant only
    print("\n" + "="*80)
    print(f"STAGE 2: Full LoRA training on best variant {best_variant}")
    print("="*80)

    best_model, best_lora_state, _ = full_train_best_variant(
        variant_idx=best_variant,
        use_hi_updates=cfg.use_hi_updates,
        system_config=system_config,
        train_loaders=train_loaders,
        val_loaders=val_loaders,
        train_sizes=train_sizes,
        client_tags=client_tags,
        cfg=cfg
    )

    # Return the best model (with LoRA mounted) and the best LoRA weights (numpy dict)
    return best_model, best_lora_state


# =============================================================================
# Example CLI usage (kept minimal)
# =============================================================================
# if __name__ == "__main__":

#     # Create config_FL_Lora with defaults (exactly your original hyperparameters)
#     config_FL_Lora = FLLoraConfig()

#     # Run the two-stage Federated LoRA pipeline
#     model, lora_state = run_federated_lora(system_config=config, config_FL_Lora=config_FL_Lora)

#     # Optional: persist the returned LoRA state (already saved inside training, too)
#     # np.savez_compressed("/tmp/best_lora_returned.npz", **lora_state)

#     # The `model` is ready to be evaluated or exported; it has the best LoRA mounted.
#     print("Done. Best LoRA is mounted on the returned model.")


In [8]:

# -*- coding: utf-8 -*-
"""
Utility: load_best_model_with_lora  (fixed + hard-min for HIsys)
- Scan lora/variant_*/meta.json to pick the best variant
- Rebuild SystemModel via mirror, load all base weights
- Attach EXTERNAL LoRA to Dense layers per meta['lora_config'] and load best_lora.npz
- (Optional) Attach softmin bias and RUL clamp to match training-time behavior
- Adds a hard-min selector for HIsys when calling `getHIandRUL`
- (Optional) Load best RUL calibration if present

Returns:
    model  : tf.keras.Model (with LoRA mounted and best LoRA weights loaded)
    info   : dict with keys {variant_idx, meta, meta_path, lora_path, calib_path, save_dir}
"""

import os
import json
import glob
from typing import Dict, Tuple, Optional, List, Any
import numpy as np
import tensorflow as tf
from tensorflow.keras import backend as K

# Reuse project constants if already defined; else fallback to defaults.
HI_UPDATE_ROOT = globals().get(
    "HI_UPDATE_ROOT",
    "/content/drive/MyDrive/Duc_an_XAI_edge_cooperation/kun/checkpoint/HI_model_update/"
)
LORA_SAVE_ROOT = globals().get("LORA_SAVE_ROOT", os.path.join(HI_UPDATE_ROOT, "lora"))


# --- Optional RUL calibration head (only if you actually use it downstream) ---
class RULCalibrationHead:
    def __init__(self, per_env: bool = False, init_a: float = 1.0, init_b: float = 0.0, init_c: float = 0.0):
        self.per_env = per_env
        if per_env:
            self.a_vars = {e: tf.Variable(init_a, tf.float32, True, name=f"rul_calib_a_env{e}") for e in [2,3,4]}
            self.b_vars = {e: tf.Variable(init_b, tf.float32, True, name=f"rul_calib_b_env{e}") for e in [2,3,4]}
            self.c_vars = {e: tf.Variable(init_c, tf.float32, True, name=f"rul_calib_c_env{e}") for e in [2,3,4]}
        else:
            self.a_var = tf.Variable(init_a, tf.float32, True, name="rul_calib_a")
            self.b_var = tf.Variable(init_b, tf.float32, True, name="rul_calib_b")
            self.c_var = tf.Variable(init_c, tf.float32, True, name="rul_calib_c")
    def load_state_dict(self, state: Dict[str, np.ndarray]):
        if self.per_env:
            for e in [2,3,4]:
                self.a_vars[e].assign(state[f"a_env{e}"])
                self.b_vars[e].assign(state[f"b_env{e}"])
                self.c_vars[e].assign(state[f"c_env{e}"])
        else:
            self.a_var.assign(state["a"]); self.b_var.assign(state["b"]); self.c_var.assign(state["c"])

def _scan_variants(lora_root: str):
    """Yield (variant_idx, meta_dict, meta_path, lora_path, calib_path_or_None)."""
    out = []
    for meta_path in glob.glob(os.path.join(lora_root, "variant_*", "meta.json")):
        vdir = os.path.dirname(meta_path)
        lora_path  = os.path.join(vdir, "best_lora.npz")
        calib_path = os.path.join(vdir, "best_rul_calib.npz")
        if not os.path.isfile(lora_path):
            continue
        with open(meta_path, "r") as f:
            meta = json.load(f)
        try:
            vidx = int(meta.get("variant_idx", os.path.basename(vdir).split("_")[-1]))
        except Exception:
            vidx = int(os.path.basename(vdir).split("_")[-1])
        out.append((vidx, meta, meta_path, lora_path, calib_path if os.path.isfile(calib_path) else None))
    if not out:
        raise RuntimeError(f"No candidate variants found under: {lora_root}")
    return out

def _choose_best(cands):
    """Lowest val_loss_weighted (fallback val_mae_weighted); tie → newer timestamp."""
    def key_fn(x):
        _, meta, _, _, _ = x
        val = float(meta.get("val_loss_weighted", meta.get("val_mae_weighted", np.inf)))
        ts  = int(meta.get("timestamp", 0))
        return (val, -ts)
    return sorted(cands, key=key_fn)[0]

# --------- Bridge wrapper so methods like getHIandRUL still work ----------
def _bridge_wrapper_methods(wrapper: tf.keras.Model, *, use_hardmin_for_hisys: bool = True, clamp_rul: bool = True):
    """
    If `wrapper` has `.base_model`, forward common methods and provide a
    bridge `getHIandRUL` that:
      - enforces HIsys = hard-min(HI_reac, HI_sep, HI_strip) when enabled
      - applies ReLU to predicted RUL when enabled
    """
    if not hasattr(wrapper, "base_model"):
        return
    base = wrapper.base_model

    # Forward a few common methods if missing on wrapper
    for name in ["predict", "save", "save_weights", "get_layer", "get_config"]:
        if hasattr(base, name) and not hasattr(wrapper, name):
            setattr(wrapper, name, getattr(base, name))

    # Clamp-aware + hard-min-aware getHIandRUL (keep tuple/list structure unchanged)
    if hasattr(base, "getHIandRUL"):
        def _getHIandRUL_bridge(*args, **kwargs):
            outs = base.getHIandRUL(*args, **kwargs)
            try:
                if isinstance(outs, (list, tuple)) and len(outs) >= 5:
                    seq = list(outs)
                    # seq indices: 0=HI_reac, 1=HI_sep, 2=HI_strip, 3=HI_sys, 4=pred_RUL, 5=hiMatrixOutput (if present)
                    if use_hardmin_for_hisys:
                        hi_reac = tf.convert_to_tensor(seq[0])
                        hi_sep  = tf.convert_to_tensor(seq[1])
                        hi_strip= tf.convert_to_tensor(seq[2])
                        # stack & reduce_min for a true hard minimum
                        hi_stack = tf.stack([hi_reac, hi_sep, hi_strip], axis=-1)  # (..., 3)
                        hi_min   = tf.reduce_min(hi_stack, axis=-1, keepdims=True)  # match column shape if inputs are column vectors
                        # reshape to original HIsys shape (robust to rank differences)
                        hi_sys_shape = tf.shape(seq[3])
                        hi_min = tf.reshape(hi_min, hi_sys_shape)
                        seq[3] = hi_min
                    if clamp_rul:
                        seq[4] = tf.nn.relu(seq[4])
                    return tuple(seq) if isinstance(outs, tuple) else seq
            except Exception:
                # if anything goes wrong, fall back to original outputs
                pass
            return outs
        setattr(wrapper, "getHIandRUL", _getHIandRUL_bridge)
        # For introspection/debug
        setattr(wrapper, "_hisys_hardmin_enabled", bool(use_hardmin_for_hisys))

# ===========================================================
# This function is used to load Lora
# -----------------------------------------------------------
def load_best_model_with_lora(  system_config: dict,
                                mount_softmin_bias: bool    = True,
                                mount_rul_clamp: bool       = True,
                                load_rul_calibration: bool  = False
                                ) -> Tuple[tf.keras.Model, Dict]:
    """
    Build SystemModel (mirror), attach LoRA, load best_lora.npz, optionally softmin/RUL clamp/calibration.
    """
    # --------------------------------------------
    # Check required helpers from your 
    # refactor are available in globals ---
    # --------------------------------------------
    required = [    "FLLoraConfig",
                    "mirror_and_build_system_model",
                    "attach_lora_to_model_external",
                    "load_lora_state_from_numpy_external",]
    
    if mount_softmin_bias:
        required.append("attach_softmin_to_model")
    if mount_rul_clamp:
        required.append("wrap_model_with_rul_clamp")
    for sym in required:
        if sym not in globals():
            raise RuntimeError(f"Missing required symbol in globals: {sym}")

    # --------------------------------------------
    # ---------- Pick best variant ---------------
    # --------------------------------------------
    cands                                       = _scan_variants(LORA_SAVE_ROOT)
    vidx, meta, meta_path, lora_npz, calib_npz  = _choose_best(cands)
    save_dir                                    = os.path.dirname(meta_path)
    print(f"[Loader] Best variant = {vidx} (meta: {meta_path})")

    # --- LoRA config from meta ---
    lora_cfg        = meta.get("lora_config", {})
    use_hi_updates  = bool(meta.get("use_hi_updates", True))
    r               = int(lora_cfg.get("r", 32))
    alpha           = float(lora_cfg.get("alpha", 64.0))
    dropout         = float(lora_cfg.get("dropout", 0.15))
    target_regex    = str(lora_cfg.get("target_regex", r".*"))

    # Assemble a minimal FLLoraConfig used for attachment/wrappers
    cfg = FLLoraConfig( lora_r              = r,
                        lora_alpha          = alpha,
                        lora_dropout        = dropout,
                        lora_target_regex   = target_regex,
                        use_softmin_bias    = bool(mount_softmin_bias),
                        enforce_rul_nonneg  = bool(mount_rul_clamp),
                    )

    # --- Rebuild SystemModel via mirror (loads ALL base weights) ---
    K.clear_session()
    model = mirror_and_build_system_model(  variant_idx     = vidx,
                                            use_hi_updates  = use_hi_updates,
                                            system_config   = system_config
                                        )

    # --- Optional: softmin bias (matches training-time inductive bias) ---
    if mount_softmin_bias:
        attach_softmin_to_model(model, cfg)

    # --- Attach EXTERNAL LoRA (Dense-only) & load best LoRA weights ---
    attach_lora_to_model_external(model, cfg)
    lora_state = dict(np.load(lora_npz))
    load_lora_state_from_numpy_external(model, lora_state)
    print(f"[Loader] Loaded LoRA weights from: {lora_npz}")

    # --- Optional: RUL clamp wrapper (ensure non-negative RUL) ---
    if mount_rul_clamp:
        model = wrap_model_with_rul_clamp(model, cfg)
        # Bridge: expose base_model APIs + hard-min HIsys + RUL clamp in getHIandRUL
        _bridge_wrapper_methods(model, use_hardmin_for_hisys=True, clamp_rul=True)
    else:
        # 即使未包 RUL clamp，也可只做硬最小 HIsys 的桥接
        _bridge_wrapper_methods(model, use_hardmin_for_hisys=True, clamp_rul=False)

    # --- Optional: RUL calibration head (if you actually consume it) ---
    if load_rul_calibration and calib_npz and os.path.isfile(calib_npz):
        calib_state = dict(np.load(calib_npz))
        rc_cfg = meta.get("rul_calib_config", {})
        per_env = bool(rc_cfg.get("per_env", False))
        init_a  = float(rc_cfg.get("init_a", 1.0))
        init_b  = float(rc_cfg.get("init_b", 0.0))
        init_c  = float(rc_cfg.get("init_c", 0.0))
        rul_calib_head = RULCalibrationHead(per_env=per_env, init_a=init_a, init_b=init_b, init_c=init_c)
        rul_calib_head.load_state_dict(calib_state)
        globals()["rul_calib_head"] = rul_calib_head
        print(f"[Loader] Loaded RUL calibration from: {calib_npz}")
    else:
        calib_npz = None  # normalize output

    info = {
        "variant_idx": vidx,
        "meta": meta,
        "meta_path": meta_path,
        "lora_path": lora_npz,
        "calib_path": calib_npz,
        "save_dir": save_dir,
    }
    return model, info


# ----------------------------
# Example minimal usage:
# ----------------------------
if __name__ == "__main__":
    # Assumes `config` dict already defined in your session for SystemModel.
    # if "config" not in globals():
    #     raise RuntimeError("Please provide your SystemModel `config` dict in globals before running this script.")
    best_model, load_info = load_best_model_with_lora(
                                                        system_config=config,
                                                        mount_softmin_bias=True,
                                                        mount_rul_clamp=True,
                                                        load_rul_calibration=False  # set True if you saved/need calibration
                                                    )
    
    print("[Done] Best model with LoRA mounted is ready.")
    print("       Variant:", load_info["variant_idx"])
    print("       Meta   :", load_info["meta_path"])
    print("       LoRA   :", load_info["lora_path"])


[Loader] Best variant = 2 (meta: ./checkpoint/flCkpt/HI_model_update/lora/variant_2/meta.json)


2025-12-01 18:28:34.625542: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2025-12-01 18:28:34.984044: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2251] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


The shape of sysHILayer (None, 8, 5)
The shape of encLastDenseDim is 800
The shape of sysHILayer (None, 8, 5)
The shape of encLastDenseDim is 800
The shape of sysHILayer (None, 8, 5)
The shape of encLastDenseDim is 800
The shape of sysHILayer (None, 8, 5)
The shape of encLastDenseDim is 800
The shape of sysHILayer (None, 8, 5)
The shape of encLastDenseDim is 800


Process:122300||2025-12-01 18:28:35,722-DEBUG [tepSystemModel.py:__init__():317]:The shape of hiSelectionInput (None, 120)
Process:122300||2025-12-01 18:28:35,761-DEBUG [tepSystemModel.py:__init__():325]:The shape of multiplied matrix (None, 3)
Process:122300||2025-12-01 18:28:35,780-DEBUG [tepSystemModel.py:__init__():335]:The shape of min, max and linear HI ((None, 1), (None, 1), (None, 1))
Process:122300||2025-12-01 18:28:35,784-DEBUG [tepSystemModel.py:__init__():338]:The shape after concatenation (None, 3)
Process:122300||2025-12-01 18:28:35,787-DEBUG [tepSystemModel.py:__init__():341]:The shape after multiplying with mask (None, 3)
Process:122300||2025-12-01 18:28:35,794-DEBUG [tepSystemModel.py:__init__():345]:The shape of HI after reducing dimension (None, 1)


The shape of sysHILayer (None, 8, 5)
The shape of encLastDenseDim is 800


Process:122300||2025-12-01 18:28:35,978-INFO [tepSystemModel.py:load_weights():896]:Loading the model at the direction: ./checkpoint/flCkpt/_tmp_variant_mirror/bestModelReac, ./checkpoint/flCkpt/_tmp_variant_mirror/bestModelSep, ./checkpoint/flCkpt/_tmp_variant_mirror/bestModelStrip, ./checkpoint/flCkpt/_tmp_variant_mirror/bestModelSys
Process:122300||2025-12-01 18:28:36,084-INFO [tepSystemModel.py:load_weights():928]:Loaded the model at the direction:./checkpoint/flCkpt/_tmp_variant_mirror/bestModelReac, ./checkpoint/flCkpt/_tmp_variant_mirror/bestModelSep, ./checkpoint/flCkpt/_tmp_variant_mirror/bestModelStrip, ./checkpoint/flCkpt/_tmp_variant_mirror/bestModelSys successfully!!!


[Softmin] Attached to 1 Dense layer(s).
[LoRA] Attached (external) to 11 Dense layer(s).
[Loader] Loaded LoRA weights from: ./checkpoint/flCkpt/HI_model_update/lora/variant_2/best_lora.npz
[Done] Best model with LoRA mounted is ready.
       Variant: 2
       Meta   : ./checkpoint/flCkpt/HI_model_update/lora/variant_2/meta.json
       LoRA   : ./checkpoint/flCkpt/HI_model_update/lora/variant_2/best_lora.npz


#--------------------------------------------------------------------

In [11]:
# Plot functions
import plotly.graph_objects as go
import os
def draw2DMultipleY(
                    xCoor       = [0,10],
                    yCoor       = [[0,10],[10,1]],
                    yLabel      = "Y Value",
                    xLabel      = "Time Unit",
                    title       = "XYAxis",
                    saveFig     = False,
                    direction   = "checkpoint/",
                    savedName   = "default",
                    useLegend   = True,
                    legend      = ["line1","line2"],
                    color       = ["blue","green"],
                    yLim       = [],):
    
    plt.figure()
    for idx,yValue in enumerate(yCoor):
        plt.plot(xCoor,yValue, color=color[idx])

    plt.title(title)
    # plt.xlabel(xLabel)
    # plt.ylabel(yLabel)

    if len(yLim) != 0:
        plt.ylim(yLim)
    if useLegend:
        plt.legend(legend, loc='upper left')
    plt.grid(True)
    if saveFig:
        
        if not os.path.exists(direction):
            os.makedirs(direction)
        plt.savefig(direction + savedName + '.png')
        plt.close()
    else:
        plt.show()


def draw2DMultipleY_plotly(
        xCoor=[0, 10],
        yCoor=[[0, 10], [10, 1]],
        yLabel="Y Value",
        xLabel="Time Unit",
        title="XYAxis",
        saveFig=False,
        direction="checkpoint/",
        savedName="default",
        useLegend=True,
        legend=["line1", "line2"],
        color=["blue", "green"],
        yLim=[]
    ):

    fig = go.Figure()

    for idx, yValue in enumerate(yCoor):
        fig.add_trace(go.Scatter(
            x=xCoor,
            y=yValue,
            mode='lines',
            name=legend[idx] if useLegend else None,
            line=dict(color=color[idx])
        ))

    # fig.update_layout(
    #     title=title,
    #     xaxis_title=xLabel,
    #     yaxis_title=yLabel,
    #     yaxis=dict(range=yLim if yLim else None),
    #     showlegend=useLegend
    # )
    fig.update_layout(
    title=title,
    xaxis_title=xLabel,
    yaxis_title=yLabel,
    yaxis=dict(range=yLim if yLim else None),
    showlegend=useLegend,
    plot_bgcolor='white',
    paper_bgcolor='white'
)


    if saveFig:
        if not os.path.exists(direction):
            os.makedirs(direction)
        fig.write_image(f"{direction}{savedName}.png")
    else:
        fig.show()


In [13]:
dfHIAll = pd.DataFrame(columns=[
                                'oriReacHI',
                                'oriSepHI',
                                'oriStripHI',
                                'oriSysHI',
                                'time'
                            ])

environmentIdx      = 4
data                = norData[environmentIdx]
trajectID           = data["Id"]
reactor             = data[preData.headline["reactor"]]
separator           = data[preData.headline["separator"]]
stripper            = data[preData.headline["stripper"]]
allSensor           = data[preData.headline["allWithoutLabelNoBias"]]
rul                 = data["rul"]
env                 = data["environmentID"]
timeTraj            = data["Time"]

hiEnv1              = {}
rulEnv1             = {}
hiEnv1["Reac"]      = []
rulEnv1["Reac"]     = []
hiEnv1["Sep"]       = []
rulEnv1["Sep"]      = []
hiEnv1["Strip"]     = []
rulEnv1["Strip"]    = []
hiEnv1["Sys"]       = []
rulEnv1["Sys"]     = []
hiEnv1["multipliedMatrix"]       = []
timeDraw            = []
hiEnv1["timeDraw"] = []


getAtIdx = 80
print(f"Getting the HI at the idx {getAtIdx}")
print(allSensor.shape)
print(env.shape)
print(timeTraj.shape)

seqXAllSensorTrain,     \
seqXRulTrain,           \
seqXTrainEnv,seqTime                = preData.getTensorData(dataList    = [allSensor,rul,env,timeTraj],
                                                            trajectId   = trajectID,  
                                                            shuffle     = False,      
                                                            getAtIdx    = getAtIdx,
                                                            seqLength   = preData.config["seqLength"],
                                                            overlap     = preData.config["overLap"]         
                                                            )

seqTimeTrain        = seqTime[:,-1,:]
curEnv              = seqXTrainEnv[:,-1,:]

print("-----------------------------------------------")
print(type(seqXAllSensorTrain))
print(seqXAllSensorTrain.shape[0])

for idxPoint in range(1000):
    # print(allSensor.shape)
    print("=-=-====================================",seqXAllSensorTrain[idxPoint:idxPoint+1,:,:].shape)
    print(seqTimeTrain[idxPoint:idxPoint+1].shape)
    print(curEnv[idxPoint:idxPoint+1].shape)

    xPredict            = [seqXAllSensorTrain[idxPoint:idxPoint+1,:,:], seqTimeTrain[idxPoint:idxPoint+1], curEnv[idxPoint:idxPoint+1]]  

    hiReac, hiSep,  hiStrip, hiSystem, predictedRul, hiMatrixOutput    = best_model.getHIandRUL(input = xPredict)
    print(hiReac)
    print(predictedRul)

#     squeezedHiReac                                 = tf.squeeze(hiReac, axis=-1)
#     squeezedHiSep                                  = tf.squeeze(hiSep, axis=-1)
#     squeezedHiStrip                                = tf.squeeze(hiStrip, axis=-1)
#     squeezedHiSys                                  = tf.squeeze(hiSystem, axis=-1)
#     squeezedRUL                                    = tf.squeeze(predictedRul,axis=-1)
#     timeDraw.append(seqTimeTrain)


#     # print(squeezedHiReac.numpy().shape)
#     # a= np.ones(squeezedHiReac.shape)*getAtIdx
#     # print(a.shape)
#     # print("ffffffffffffff")
#     # b = np.squeeze(seqTimeTrain)
#     # print(b.shape)


#     dfTemp      = pd.DataFrame({'id'           : np.ones(squeezedHiReac.shape)*getAtIdx,
#                                 'oriReacHI'    : squeezedHiReac.numpy(), 
#                                 'oriSepHI'     : squeezedHiSep.numpy(),
#                                 'oriStripHI'   : squeezedHiStrip.numpy(),
#                                 'oriSysHI'     : squeezedHiSys.numpy(),
#                                 'rul'          : squeezedRUL.numpy(), 
#                                 'time'      : np.squeeze(seqTimeTrain[0:1])})


#     dfHIAll     = pd.concat([dfHIAll,dfTemp],ignore_index=True)



#     dfHIAll["reacHI"]   = dfHIAll["oriReacHI"].copy()
#     dfHIAll["sepHI"]    = dfHIAll["oriSepHI"].copy()
#     dfHIAll["stripHI"]  = dfHIAll["oriStripHI"].copy()



    # hiEnv1["Reac"].append(hiReac)
    # hiEnv1["Sep"].append(hiSep)
    # hiEnv1["Strip"].append(hiStrip)
    # hiEnv1["Sys"].append(hiSystem)
    # hiEnv1["timeDraw"].append(seqTimeTrain)
    # hiEnv1["multipliedMatrix"].append(hiMatrixOutput)
    # rulEnv1["Sys"].append(predictedRul)
    # rulEnv1["Reac"].append(squeezedRulReac)
    # rulEnv1["Sep"].append(squeezedRulSep)
    # rulEnv1["Strip"].append(squeezedRulStrip)

    # break
    


# if normalizeHI:
#     [hiEnv1["Reac"], hiEnv1["Sep"], hiEnv1["Strip"],hiEnv1["Sys"]], timeDraw , rulEnv1["Sys"], hiEnv1["multipliedMatrix"]= preData.normalizeHIInGeneral(inData = [hiEnv1["Reac"], hiEnv1["Sep"], hiEnv1["Strip"],hiEnv1["Sys"]], time= timeDraw, rul = rulEnv1["Sys"], multipliedMt = hiEnv1["multipliedMatrix"], indexToShutdown = 2, showRedundantShutdown = showRedundantShutdown)
# hiEnv1["Sys"]                                  = preData.normalizeHIInGeneral(inData = [hiEnv1["Reac"], hiEnv1["Sep"], hiEnv1["Strip"]], indexToShutdown = 2)


Process:122300||2025-12-01 18:38:22,797-INFO [preProcessingDataTEP.py:getTensorData():872]:Spliting data into sequences now ...


Getting the HI at the idx 80
(122808, 45)
(122808,)
(122808,)
-----------------------------------------------
<class 'numpy.ndarray'>
1059
=-=-==================================== (1, 128, 45)
(1, 1)
(1, 1)
tf.Tensor([[-0.7628069]], shape=(1, 1), dtype=float32)
tf.Tensor([[67.46642]], shape=(1, 1), dtype=float32)
=-=-==================================== (1, 128, 45)
(1, 1)
(1, 1)
tf.Tensor([[-1.1012398]], shape=(1, 1), dtype=float32)
tf.Tensor([[67.67299]], shape=(1, 1), dtype=float32)
=-=-==================================== (1, 128, 45)
(1, 1)
(1, 1)
tf.Tensor([[-0.949739]], shape=(1, 1), dtype=float32)
tf.Tensor([[67.85756]], shape=(1, 1), dtype=float32)
=-=-==================================== (1, 128, 45)
(1, 1)
(1, 1)
tf.Tensor([[-0.74501705]], shape=(1, 1), dtype=float32)
tf.Tensor([[67.9947]], shape=(1, 1), dtype=float32)
=-=-==================================== (1, 128, 45)
(1, 1)
(1, 1)
tf.Tensor([[-0.55694926]], shape=(1, 1), dtype=float32)
tf.Tensor([[67.95375]], shape=(1, 

KeyboardInterrupt: 